# HW03 — Model Serving & Deployment

Your model is trained and tracked in MLflow. A model that only runs in a notebook is not useful in production.

In this homework you will:

- verify that your trained model is reproducible and ready for serving
- wrap it in a FastAPI service with proper input validation and batch support
- measure the performance difference between single and batch prediction
- package the service in Docker with a size-optimized image
- write Kubernetes manifests to deploy the service

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords, tokens, or notebook checkpoints.
Do not hardcode passwords anywhere in your code.

## Useful references

- MLflow model loading: https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html
- FastAPI: https://fastapi.tiangolo.com/
- FastAPI lifespan: https://fastapi.tiangolo.com/advanced/events/
- Pydantic v2: https://docs.pydantic.dev/latest/
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker multi-stage builds: https://docs.docker.com/build/building/multi-stage/
- Kubernetes Deployments: https://kubernetes.io/docs/concepts/workloads/controllers/deployment/
- Kubernetes Services: https://kubernetes.io/docs/concepts/services-networking/service/

## What to avoid

- Loading the model inside the request handler. Load once at startup.
- Hardcoded passwords in any source file.
- A Docker image that bakes in the model file. Pull from MLflow at startup.
- Returning raw numpy types from the API. JSON needs native Python types.
- Skipping the batch vs single benchmark. The numbers tell a story.

In [1]:
import os
import re
import time
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

PROJECT = Path.cwd()
for path in ['src/airbnb_serving', 'k8s', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_serving/__init__.py').write_text('__version__ = "0.1.0"\n')

STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
EXPERIMENT_NAME = f'qbc12_hw02_{safe_student}'

print('PROJECT:', PROJECT)
print('EXPERIMENT_NAME:', EXPERIMENT_NAME)

PROJECT: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A
EXPERIMENT_NAME: qbc12_hw02_student_alireza_abouei


---
## Part 1 — Model Reproducibility Check

Before serving a model, you must verify it produces exactly the same output as it did during training.

This is called a **reproducibility check** and it catches silent bugs like:
- preprocessing mismatch between training and serving
- wrong model version loaded
- feature column order changed

### 1.1 Connect to MLflow and load your best model

In [2]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://185.50.38.163:33014')
MLFLOW_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME', '') or input('MLflow username: ').strip()
MLFLOW_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD', '') or input('MLflow password: ').strip()

os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f'Experiment not found: {EXPERIMENT_NAME}. Complete HW02 first.')

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.leakage_status = 'clean' and tags.selected_for_serving = 'true'",
    order_by=['metrics.f1 DESC'],
)

if runs.empty:
    raise ValueError(
        'No run tagged selected_for_serving=true found. '
        'Go to MLflow UI, find your best clean run, and add the tag.'
    )

BEST_RUN_ID = runs.iloc[0]['run_id']
MODEL_URI = f'runs:/{BEST_RUN_ID}/model'

print('Best run ID  :', BEST_RUN_ID)
print('Model URI    :', MODEL_URI)
print('Run name     :', runs.iloc[0].get('tags.mlflow.runName'))
print('F1 score     :', runs.iloc[0].get('metrics.f1'))

Best run ID  : 2f4885cc6b2b42baa62c279937ea4401
Model URI    : runs:/2f4885cc6b2b42baa62c279937ea4401/model
Run name     : v5_random_forest
F1 score     : 0.9884048887496083


In [3]:
import joblib
from pathlib import Path

try:
    model = mlflow.sklearn.load_model(MODEL_URI)
    model_load_mode = "mlflow_online"
    print("Loaded model from MLflow.")

except Exception as e:
    print("MLflow online load failed.")
    print("Reason:", repr(e))
    print("Trying local fallback model...")

    candidate_paths = [
        Path("mlflow_artifacts/v5_random_forest/model.joblib"),
        Path("HW03/mlflow_artifacts/v5_random_forest/model.joblib"),
        Path("/mlflow_artifacts/v5_random_forest/model.joblib"),
    ]

    local_model_path = next((path for path in candidate_paths if path.exists()), None)

    if local_model_path is None:
        raise FileNotFoundError(
            "Local fallback model not found. Tried:\n"
            + "\n".join(str(path) for path in candidate_paths)
        )

    model = joblib.load(local_model_path)
    model_load_mode = "local_joblib_fallback"
    print("Loaded local fallback model from:", local_model_path)

print("Model load mode:", model_load_mode)
print("Model type:", type(model))
print(
    "Model steps:",
    list(model.named_steps.keys()) if hasattr(model, "named_steps") else "not a pipeline",
)

MLflow online load failed.
Reason: MlflowException("The following failures occurred while downloading one or more artifacts from http://185.50.38.163:33014/api/2.0/mlflow-artifacts/artifacts/10/2f4885cc6b2b42baa62c279937ea4401/artifacts:\n##### File model #####\nAPI request to http://185.50.38.163:33014/api/2.0/mlflow-artifacts/artifacts/10/2f4885cc6b2b42baa62c279937ea4401/artifacts/model failed with exception HTTPConnectionPool(host='185.50.38.163', port=33014): Max retries exceeded with url: /api/2.0/mlflow-artifacts/artifacts/10/2f4885cc6b2b42baa62c279937ea4401/artifacts/model (Caused by ResponseError('too many 500 error responses'))")
Trying local fallback model...
Loaded local fallback model from: mlflow_artifacts/v5_random_forest/model.joblib
Model load mode: local_joblib_fallback
Model type: <class 'sklearn.pipeline.Pipeline'>
Model steps: ['preprocessor', 'model']


### 1.2 Load your HW01 feature dataset

You will use a small sample from your HW01 feature parquet file to verify reproducibility.

In [4]:
FEATURE_COLS = [
    'room_type', 'property_type', 'neighbourhood_name',
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price',
    'minimum_nights', 'maximum_nights', 'instant_bookable', 'host_is_superhost',
    'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff',
    'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff',
    'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d',
    'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d',
    'available_days_last_30d', 'available_rate_last_30d',
    'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d',
]
TARGET_COL = 'high_demand_proxy'

# Load your HW01 parquet file
# Adjust the path if needed
parquet_path = list(Path('data/features').glob('*.parquet'))
if not parquet_path:
    raise FileNotFoundError('HW01 feature parquet not found. Run HW01 ETL first.')

df = pd.read_parquet(parquet_path[0])
print('Dataset shape:', df.shape)
df[FEATURE_COLS + [TARGET_COL]].head(3)

Dataset shape: (10480, 35)


,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,...,days_since_last_review,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,high_demand_proxy
0,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,356,...,338.0,0,0.000000,3.0,30.0,0,0.000000,3.0,30.0,1
1,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,730,...,338.0,45,0.500000,2.0,730.0,14,0.466667,2.0,730.0,0
2,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,730,...,337.0,42,0.466667,2.0,730.0,16,0.533333,2.0,730.0,1


### 1.3 Reproducibility check

**TODO 1.3**

Take a sample of 50 rows from the dataset.

Run `model.predict()` on those rows **twice** and verify the results are identical.

Then compare the predictions against the `high_demand_proxy` column and print:
- how many predictions match the training label
- the accuracy on this sample

If both runs produce identical output, print `Reproducibility check passed.`
If they differ, raise a `ValueError`.

In [5]:
# TODO 1.3
# Reproducibility check:
# The same model and same input sample should produce identical predictions.

sample = df[FEATURE_COLS + [TARGET_COL]].sample(50, random_state=42).copy()

X_sample = sample[FEATURE_COLS].copy()
y_sample = sample[TARGET_COL].astype(int)

# Add model-side feature if the notebook/API-side feature is present.
# This does not change the original dataset; it only prepares input for the model.
if "host_is_superhost" in X_sample.columns and "is_superhost" not in X_sample.columns:
    X_sample["is_superhost"] = X_sample["host_is_superhost"].astype(bool)

# Add derived feature if it is not already part of X_sample.
if "has_reviews_before_cutoff" not in X_sample.columns:
    X_sample["has_reviews_before_cutoff"] = (
        X_sample["total_reviews_before_cutoff"].fillna(0).astype(float) > 0
    )

expected_cols = None

if hasattr(model, "feature_names_in_"):
    expected_cols = list(model.feature_names_in_)
elif hasattr(model, "named_steps"):
    for step in model.named_steps.values():
        if hasattr(step, "feature_names_in_"):
            expected_cols = list(step.feature_names_in_)
            break

if expected_cols is not None:
    missing_cols = [col for col in expected_cols if col not in X_sample.columns]
    if missing_cols:
        raise ValueError(f"Missing model input columns: {missing_cols}")

    X_sample = X_sample[expected_cols]

preds_first = model.predict(X_sample)
preds_second = model.predict(X_sample)

if not np.array_equal(preds_first, preds_second):
    raise ValueError("Reproducibility check failed: predictions are not identical.")

matches = int((preds_first == y_sample.to_numpy()).sum())
sample_accuracy = matches / len(y_sample)

print("Sample size:", len(sample))
print("Matching predictions:", matches)
print("Sample accuracy:", round(sample_accuracy, 4))
print("Unique predictions:", sorted(pd.Series(preds_first).unique().tolist()))
print("Reproducibility check passed.")

Sample size: 50
Matching predictions: 50
Sample accuracy: 1.0
Unique predictions: [0, 1]
Reproducibility check passed.


---
## Part 2 — FastAPI Service

A REST API is the standard way to expose an ML model to other systems.

You will build a FastAPI app with two prediction endpoints:
- `POST /predict` — single listing prediction
- `POST /predict/batch` — multiple listings in one request

Then you will measure how much faster batch is compared to calling single predict in a loop.

### 2.1 Input and output schemas

In [6]:
# TODO 2.1
# Create src/airbnb_serving/schema.py

(PROJECT / "src/airbnb_serving/schema.py").write_text(textwrap.dedent("""
from typing import Optional

from pydantic import BaseModel, ConfigDict, Field


class ListingFeatures(BaseModel):
    model_config = ConfigDict(extra="forbid")

    room_type: str
    property_type: str
    neighbourhood_name: str

    accommodates: int = Field(ge=0)
    bedrooms: Optional[float] = Field(default=None, ge=0)
    beds: Optional[float] = Field(default=None, ge=0)
    bathrooms: Optional[float] = Field(default=None, ge=0)

    listing_price: float = Field(ge=0)
    minimum_nights: int = Field(ge=0)
    maximum_nights: int = Field(ge=0)

    instant_bookable: bool
    host_is_superhost: bool

    host_listing_count: int = Field(ge=0)

    total_reviews_before_cutoff: int = Field(ge=0)
    unique_reviewers_before_cutoff: int = Field(ge=0)

    avg_comment_len_before_cutoff: Optional[float] = Field(default=None, ge=0)
    max_comment_len_before_cutoff: Optional[float] = Field(default=None, ge=0)
    days_since_last_review: Optional[float] = Field(default=None, ge=0)

    available_days_last_90d: int = Field(ge=0)
    available_rate_last_90d: float = Field(ge=0, le=1)
    avg_minimum_nights_calendar_last_90d: Optional[float] = Field(default=None, ge=0)
    avg_maximum_nights_calendar_last_90d: Optional[float] = Field(default=None, ge=0)

    available_days_last_30d: int = Field(ge=0)
    available_rate_last_30d: float = Field(ge=0, le=1)
    avg_minimum_nights_calendar_last_30d: Optional[float] = Field(default=None, ge=0)
    avg_maximum_nights_calendar_last_30d: Optional[float] = Field(default=None, ge=0)


class PredictionResponse(BaseModel):
    listing_id: Optional[int] = None
    prediction: int = Field(ge=0, le=1)
    probability_high_demand: float = Field(ge=0, le=1)
    model_run_id: str
""").strip() + "\n")

print("Created:", PROJECT / "src/airbnb_serving/schema.py")

Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/src/airbnb_serving/schema.py


### 2.2 Prediction logic

In [7]:
# TODO 2.2
# Create src/airbnb_serving/predictor.py

(PROJECT / "src/airbnb_serving/predictor.py").write_text(textwrap.dedent("""
from typing import Iterable

import numpy as np
import pandas as pd

from .schema import ListingFeatures, PredictionResponse


def _get_expected_columns(model) -> list[str] | None:
    \"\"\"Infer the feature columns expected by the trained sklearn model/pipeline.\"\"\"
    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)

    if hasattr(model, "named_steps"):
        for step in model.named_steps.values():
            if hasattr(step, "feature_names_in_"):
                return list(step.feature_names_in_)

    return None


def _get_positive_class_index(model) -> int:
    \"\"\"Find the probability column for positive class 1.\"\"\"
    classes = getattr(model, "classes_", None)

    if classes is None and hasattr(model, "named_steps"):
        for step in reversed(list(model.named_steps.values())):
            if hasattr(step, "classes_"):
                classes = step.classes_
                break

    if classes is None:
        return 1

    classes = list(classes)
    if 1 in classes:
        return classes.index(1)

    return len(classes) - 1


def _features_to_dataframe(features_list: Iterable[ListingFeatures], model) -> pd.DataFrame:
    \"\"\"Convert validated Pydantic inputs into the model input DataFrame.\"\"\"
    rows = [item.model_dump() for item in features_list]

    if not rows:
        raise ValueError("At least one listing is required for prediction.")

    df = pd.DataFrame(rows)

    # Public API field -> model training field.
    if "host_is_superhost" in df.columns and "is_superhost" not in df.columns:
        df["is_superhost"] = df["host_is_superhost"].astype(bool)

    # Internal derived feature expected by the trained model.
    if "has_reviews_before_cutoff" not in df.columns:
        df["has_reviews_before_cutoff"] = (
            df["total_reviews_before_cutoff"].fillna(0).astype(float) > 0
        )

    expected_columns = _get_expected_columns(model)

    if expected_columns is not None:
        missing_columns = [col for col in expected_columns if col not in df.columns]
        if missing_columns:
            raise ValueError(f"Missing model input columns: {missing_columns}")

        return df[expected_columns]

    return df


def _predict_probabilities(model, X: pd.DataFrame) -> np.ndarray:
    \"\"\"Return positive-class probabilities if available; otherwise use predictions.\"\"\"
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        positive_idx = _get_positive_class_index(model)
        return proba[:, positive_idx]

    predictions = model.predict(X)
    return predictions.astype(float)


def predict_single(
    features: ListingFeatures,
    model,
    run_id: str,
) -> PredictionResponse:
    \"\"\"Predict high-demand probability for one listing.\"\"\"
    X = _features_to_dataframe([features], model)

    predictions = model.predict(X)
    probabilities = _predict_probabilities(model, X)

    prediction = int(predictions[0])
    probability = float(probabilities[0])

    return PredictionResponse(
        listing_id=None,
        prediction=prediction,
        probability_high_demand=probability,
        model_run_id=str(run_id),
    )


def predict_batch(
    features_list: list[ListingFeatures],
    model,
    run_id: str,
) -> list[PredictionResponse]:
    \"\"\"Predict high-demand probability for a batch of listings.

    The model is called once for the full batch.
    \"\"\"
    X = _features_to_dataframe(features_list, model)

    predictions = model.predict(X)
    probabilities = _predict_probabilities(model, X)

    responses: list[PredictionResponse] = []

    for prediction, probability in zip(predictions, probabilities):
        responses.append(
            PredictionResponse(
                listing_id=None,
                prediction=int(prediction),
                probability_high_demand=float(probability),
                model_run_id=str(run_id),
            )
        )

    return responses
""").strip() + "\n")

print("Created:", PROJECT / "src/airbnb_serving/predictor.py")

Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/src/airbnb_serving/predictor.py


### 2.3 FastAPI app

In [8]:
# TODO 2.3
# Create src/airbnb_serving/app.py

(PROJECT / "src/airbnb_serving/app.py").write_text(textwrap.dedent("""
import os
from pathlib import Path
from typing import List

import joblib
import mlflow
import mlflow.sklearn
from fastapi import FastAPI, HTTPException, status

from .predictor import predict_batch, predict_single
from .schema import ListingFeatures, PredictionResponse


DEFAULT_RUN_ID = "2f4885cc6b2b42baa62c279937ea4401"
DEFAULT_EXPERIMENT_NAME = "qbc12_hw02_student_alireza_abouei"
DEFAULT_LOCAL_MODEL_PATH = "mlflow_artifacts/v5_random_forest/model.joblib"


RUN_ID = os.getenv("MLFLOW_RUN_ID", DEFAULT_RUN_ID)
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", DEFAULT_EXPERIMENT_NAME)
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://185.50.38.163:33014")
MODEL_URI = os.getenv("MODEL_URI", f"runs:/{RUN_ID}/model")
LOCAL_MODEL_PATH = os.getenv("LOCAL_MODEL_PATH", DEFAULT_LOCAL_MODEL_PATH)

model = None
model_load_mode = None
model_load_error = None


def load_model():
    \"\"\"Load the selected model.

    First tries MLflow online loading. If that fails, it falls back to a local joblib
    model artifact. The local fallback is useful when the MLflow artifact server is
    slow or temporarily returns 500 errors.
    \"\"\"
    global model, model_load_mode, model_load_error

    if model is not None:
        return model

    mlflow.set_tracking_uri(TRACKING_URI)

    try:
        model = mlflow.sklearn.load_model(MODEL_URI)
        model_load_mode = "mlflow_online"
        model_load_error = None
        return model

    except Exception as online_error:
        local_path = Path(LOCAL_MODEL_PATH)

        if not local_path.exists():
            model_load_error = (
                "MLflow loading failed and local fallback model was not found. "
                f"MLflow error: {online_error!r}. "
                f"Local path tried: {local_path}"
            )
            raise RuntimeError(model_load_error) from online_error

        try:
            model = joblib.load(local_path)
            model_load_mode = "local_joblib_fallback"
            model_load_error = None
            return model

        except Exception as local_error:
            model_load_error = (
                "Both MLflow loading and local joblib fallback loading failed. "
                f"MLflow error: {online_error!r}. "
                f"Local error: {local_error!r}"
            )
            raise RuntimeError(model_load_error) from local_error


app = FastAPI(
    title="Airbnb Demand Prediction API",
    description="Serve the selected HW02 classic ML model with FastAPI.",
    version="1.0.0",
)


@app.on_event("startup")
def startup_event():
    \"\"\"Try loading the model during startup.

    If loading fails, the API still starts so /health can show the error.
    \"\"\"
    try:
        load_model()
    except Exception:
        # Keep the service alive. /health will report the loading problem.
        pass


@app.get("/")
def root():
    return {
        "service": "airbnb-demand-prediction",
        "status": "running",
        "docs_url": "/docs",
        "health_url": "/health",
    }


@app.get("/health")
def health():
    return {
        "status": "ok" if model is not None else "error",
        "model_loaded": model is not None,
        "model_load_mode": model_load_mode,
        "model_run_id": RUN_ID,
        "model_uri": MODEL_URI,
        "error": model_load_error,
    }


@app.get("/model-info")
def model_info():
    return {
        "model_run_id": RUN_ID,
        "experiment_name": EXPERIMENT_NAME,
        "model_uri": MODEL_URI,
        "tracking_uri": TRACKING_URI,
        "model_load_mode": model_load_mode,
        "local_model_path": LOCAL_MODEL_PATH,
    }


@app.post("/predict", response_model=PredictionResponse)
def predict_endpoint(features: ListingFeatures):
    try:
        loaded_model = load_model()
        return predict_single(features, loaded_model, RUN_ID)

    except Exception as exc:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Prediction failed: {exc}",
        ) from exc


@app.post("/predict-batch", response_model=List[PredictionResponse])
def predict_batch_endpoint(features: List[ListingFeatures]):
    if not features:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Batch request must contain at least one listing.",
        )

    try:
        loaded_model = load_model()
        return predict_batch(features, loaded_model, RUN_ID)

    except Exception as exc:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Batch prediction failed: {exc}",
        ) from exc
""").strip() + "\n")

print("Created:", PROJECT / "src/airbnb_serving/app.py")

Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/src/airbnb_serving/app.py


### 2.4 Package metadata

In [9]:
# TODO 2.4
# Create packaging and dependency files for the FastAPI service.

(PROJECT / "src/airbnb_serving/__init__.py").write_text(
    '__version__ = "1.0.0"\n'
)

(PROJECT / "requirements.txt").write_text(textwrap.dedent("""
fastapi>=0.110,<1.0
uvicorn[standard]>=0.27,<1.0
pydantic>=2.0,<3.0
pandas>=2.0,<3.0
numpy>=1.24,<3.0
scikit-learn>=1.3,<2.0
mlflow>=2.0,<4.0
joblib>=1.3,<2.0
python-dotenv>=1.0,<2.0
pyarrow>=14.0,<22.0
""").strip() + "\n")

(PROJECT / "pyproject.toml").write_text(textwrap.dedent("""
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "airbnb-serving"
version = "1.0.0"
description = "FastAPI service for serving the selected Airbnb demand prediction model."
requires-python = ">=3.10"
dependencies = [
    "fastapi>=0.110,<1.0",
    "uvicorn[standard]>=0.27,<1.0",
    "pydantic>=2.0,<3.0",
    "pandas>=2.0,<3.0",
    "numpy>=1.24,<3.0",
    "scikit-learn>=1.3,<2.0",
    "mlflow>=2.0,<4.0",
    "joblib>=1.3,<2.0",
    "python-dotenv>=1.0,<2.0",
    "pyarrow>=14.0,<22.0"
]

[tool.setuptools.packages.find]
where = ["src"]
""").strip() + "\n")

print("Created:", PROJECT / "src/airbnb_serving/__init__.py")
print("Created:", PROJECT / "requirements.txt")
print("Created:", PROJECT / "pyproject.toml")

Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/src/airbnb_serving/__init__.py
Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/requirements.txt
Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/pyproject.toml


### 2.5 Local install and smoke test

Install the package and manually start the server in a terminal before running the test cell below.

```bash
pip install -e .

MODEL_RUN_ID=<your_run_id> \
MLFLOW_TRACKING_URI=http://185.50.38.163:33014 \
MLFLOW_TRACKING_USERNAME=<user> \
MLFLOW_TRACKING_PASSWORD=<pass> \
uvicorn airbnb_serving.app:app --host 0.0.0.0 --port 8000
```

In [10]:
import sys
!{sys.executable} -m pip install -q -e .


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [33]:
import subprocess, time, signal, os

server_proc = subprocess.Popen(
    ['uvicorn', 'airbnb_serving.app:app', '--host', '0.0.0.0', '--port', '8001'],
    env={**os.environ, 'MODEL_RUN_ID': BEST_RUN_ID},
)
time.sleep(10)  # wait for model to load from MLflow
print('Server started, PID:', server_proc.pid)

INFO:     Started server process [52744]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


Server started, PID: 52744


In [34]:
import requests

BASE_URL = 'http://localhost:8001'

health = requests.get(f'{BASE_URL}/health')
assert health.status_code == 200, f'Health check failed: {health.text}'
print('Health:', health.json())

sample_payload = {
    'room_type': 'Entire home/apt',
    'property_type': 'Entire rental unit',
    'neighbourhood_name': 'Centrum-West',
    'accommodates': 2,
    'bedrooms': 1.0,
    'beds': 1.0,
    'bathrooms': 1.0,
    'listing_price': 150.0,
    'minimum_nights': 2,
    'maximum_nights': 365,
    'instant_bookable': True,
    'host_is_superhost': False,
    'host_listing_count': 1,
    'total_reviews_before_cutoff': 10.0,
    'unique_reviewers_before_cutoff': 9.0,
    'avg_comment_len_before_cutoff': 120.0,
    'max_comment_len_before_cutoff': 300.0,
    'days_since_last_review': 30.0,
    'available_days_last_90d': 45,
    'available_rate_last_90d': 0.5,
    'avg_minimum_nights_calendar_last_90d': 2.0,
    'avg_maximum_nights_calendar_last_90d': 365.0,
    'available_days_last_30d': 15,
    'available_rate_last_30d': 0.5,
    'avg_minimum_nights_calendar_last_30d': 2.0,
    'avg_maximum_nights_calendar_last_30d': 365.0,
}

resp = requests.post(f'{BASE_URL}/predict', json=sample_payload)
assert resp.status_code == 200, f'Single predict failed: {resp.text}'
print('Single predict:', resp.json())

print('Local smoke test passed.')

INFO:     127.0.0.1:51402 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok', 'model_loaded': True, 'model_load_mode': 'local_joblib_fallback', 'model_run_id': '2f4885cc6b2b42baa62c279937ea4401', 'model_uri': 'runs:/2f4885cc6b2b42baa62c279937ea4401/model', 'error': None}
INFO:     127.0.0.1:51414 - "POST /predict HTTP/1.1" 200 OK
Single predict: {'listing_id': None, 'prediction': 0, 'probability_high_demand': 0.08823096090623848, 'model_run_id': '2f4885cc6b2b42baa62c279937ea4401'}
Local smoke test passed.


### 2.6 Batch vs single benchmark

**TODO 2.6**

Take 100 rows from your feature dataset.

Measure:
1. Time to call `POST /predict` 100 times in a loop (single)
2. Time to call `POST /predict/batch` once with all 100 rows (batch)

Print a comparison table with total time and time per prediction for each approach.

Then answer: why is batch faster? Write your answer as a comment in the cell.

In [35]:
# TODO 2.6
# Benchmark single prediction requests vs one batch prediction request.
# The API service must already be running before this cell is executed.

import math
import time
import requests
import pandas as pd
import numpy as np

API_BASE_URL = "http://127.0.0.1:8001"
BENCHMARK_SIZE = 100


def clean_for_json(row: dict) -> dict:
    cleaned = {}

    for key, value in row.items():
        if pd.isna(value):
            cleaned[key] = None
        elif hasattr(value, "item"):
            cleaned[key] = value.item()
        else:
            cleaned[key] = value

    return cleaned


required_api_columns = [
    "room_type",
    "property_type",
    "neighbourhood_name",
    "accommodates",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "host_is_superhost",
    "host_listing_count",
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
    "available_days_last_90d",
    "available_rate_last_90d",
    "available_days_last_30d",
    "available_rate_last_30d",
]

valid_df = df[FEATURE_COLS].dropna(subset=required_api_columns).head(BENCHMARK_SIZE)

if len(valid_df) < BENCHMARK_SIZE:
    print(f"Only {len(valid_df)} fully valid rows found for benchmarking.")

benchmark_rows = [
    clean_for_json(row)
    for row in valid_df.to_dict(orient="records")
]

health_response = requests.get(f"{API_BASE_URL}/health", timeout=10)
health_response.raise_for_status()

print("Health response:")
print(health_response.json())


# 1. Single prediction endpoint called many times.
single_start = time.perf_counter()

single_responses = []

for i, row in enumerate(benchmark_rows):
    response = requests.post(
        f"{API_BASE_URL}/predict",
        json=row,
        timeout=30,
    )

    if response.status_code != 200:
        print(f"Single request failed at row index {i}")
        print("Payload:")
        print(row)
        print("Response:")
        print(response.text)

    response.raise_for_status()
    single_responses.append(response.json())

single_total = time.perf_counter() - single_start
single_per_prediction_ms = (single_total / len(benchmark_rows)) * 1000


# 2. Batch prediction endpoint called once.
batch_start = time.perf_counter()

batch_response = requests.post(
    f"{API_BASE_URL}/predict-batch",
    json=benchmark_rows,
    timeout=60,
)

if batch_response.status_code != 200:
    print("Batch response error:")
    print(batch_response.text)

batch_response.raise_for_status()
batch_predictions = batch_response.json()

batch_total = time.perf_counter() - batch_start
batch_per_prediction_ms = (batch_total / len(benchmark_rows)) * 1000


assert len(single_responses) == len(benchmark_rows)
assert len(batch_predictions) == len(benchmark_rows)

comparison = pd.DataFrame(
    [
        {
            "method": "single loop",
            "request_count": len(benchmark_rows),
            "prediction_count": len(single_responses),
            "total_seconds": round(single_total, 4),
            "per_prediction_ms": round(single_per_prediction_ms, 4),
        },
        {
            "method": "batch",
            "request_count": 1,
            "prediction_count": len(batch_predictions),
            "total_seconds": round(batch_total, 4),
            "per_prediction_ms": round(batch_per_prediction_ms, 4),
        },
    ]
)

speedup = single_total / batch_total if batch_total > 0 else float("inf")

print("\nBenchmark comparison:")
print(comparison.to_string(index=False))

print(f"\nBatch speedup: {speedup:.2f}x")

INFO:     127.0.0.1:51420 - "GET /health HTTP/1.1" 200 OK
Health response:
{'status': 'ok', 'model_loaded': True, 'model_load_mode': 'local_joblib_fallback', 'model_run_id': '2f4885cc6b2b42baa62c279937ea4401', 'model_uri': 'runs:/2f4885cc6b2b42baa62c279937ea4401/model', 'error': None}
INFO:     127.0.0.1:51434 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51440 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51450 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51464 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51478 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51490 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51504 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51516 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51530 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36152 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36160 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36172 - "POST /predict HTTP/1.1" 2

In [36]:
server_proc.terminate()
print('Server stopped.')

Server stopped.


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [52744]


---
## Part 3 — Docker

You will build two Docker images and compare their sizes.

This teaches you that image size is not free — it affects pull time, storage cost, and attack surface.

### 3.1 Naive Dockerfile

In [37]:
# TODO 3.1
# Create Dockerfile.naive
#
# This is intentionally simple and unoptimized.
# It copies the whole project into the image, installs dependencies,
# installs the local package, and runs the FastAPI app.

(PROJECT / "Dockerfile.naive").write_text(textwrap.dedent("""
FROM python:3.11

WORKDIR /app

COPY . .

RUN pip install --no-cache-dir -r requirements.txt \\
    && pip install --no-cache-dir -e .

EXPOSE 8000

CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]
""").strip() + "\n")

print("Created:", PROJECT / "Dockerfile.naive")

Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/Dockerfile.naive


### 3.2 Optimized Dockerfile

A multi-stage build separates the build environment from the runtime environment.

Stage 1 (builder): install everything, build the package.
Stage 2 (runtime): copy only what is needed to run, nothing else.

In [11]:
# TODO 3.2
# Create optimized Dockerfile and .dockerignore.
#
# Compared with Dockerfile.naive:
# - uses python:3.11-slim instead of full python image
# - copies dependency files first for better Docker layer caching
# - avoids copying virtualenvs, notebooks, data, logs, and local cache files
# - installs package after dependencies
# - runs uvicorn on 0.0.0.0 so Docker port mapping works

(PROJECT / "Dockerfile").write_text(textwrap.dedent("""
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV PIP_NO_CACHE_DIR=1

WORKDIR /app

RUN apt-get update \\
    && apt-get install -y --no-install-recommends curl \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt pyproject.toml ./
COPY src ./src

RUN pip install --upgrade pip \\
    && pip install -r requirements.txt \\
    && pip install -e .

EXPOSE 8000

CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]
""").strip() + "\n")


(PROJECT / ".dockerignore").write_text(textwrap.dedent("""
# Python cache
__pycache__/
*.py[cod]
*.pyo
*.pyd
.pytest_cache/
.mypy_cache/
.ruff_cache/

# Virtual environments
.venv/
venv/
env/

# Git / IDE
.git/
.gitignore
.idea/
.vscode/

# Notebook / OS files
.ipynb_checkpoints/
.DS_Store

# Notebooks are not required inside the serving image
*.ipynb

# Local secrets
.env

# Local logs
*.log
logs/

# Local MLflow folders
mlruns/
mlartifacts/

# Local datasets and large files
data/

# Local model fallback artifacts
# The Docker service should load from MLflow unless LOCAL_MODEL_PATH is explicitly mounted.
mlflow_artifacts/
artifacts/

# Assignment output folders
outputs/
screenshots/
""").strip() + "\n")

print("Created:", PROJECT / "Dockerfile")
print("Created:", PROJECT / ".dockerignore")

Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/Dockerfile
Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/.dockerignore


### 3.3 Build and compare image sizes

In [21]:
!docker build -f Dockerfile.naive -t qbc12-airbnb-serving:naive .
!docker build -f Dockerfile -t qbc12-airbnb-serving:optimized .


[+] Building 0.0s (0/1)                                    docker:desktop-linux
[+] Building 0.2s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile.naive                 0.0s
 => => transferring dockerfile: 274B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11             0.1s
[+] Building 0.3s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile.naive                 0.0s
 => => transferring dockerfile: 274B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11             0.3s
[+] Building 0.5s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile.naive                 0.0s
 => => transferring dockerfile: 274B                                       0.0s
 => [internal] load metadata for docker

In [22]:
# TODO 3.3
# Compare Docker image sizes and write the required report file.

import json
import subprocess
from pathlib import Path

import pandas as pd

REPORT_DIR = PROJECT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

images = {
    "naive": "airbnb-serving:naive",
    "optimized": "airbnb-serving:optimized",
}


def get_image_size_bytes(image_name: str) -> int | None:
    try:
        result = subprocess.run(
            ["docker", "image", "inspect", image_name, "--format", "{{json .Size}}"],
            capture_output=True,
            text=True,
            check=True,
        )
        return int(json.loads(result.stdout.strip()))
    except Exception as exc:
        print(f"Could not inspect image {image_name}: {exc}")
        return None


def bytes_to_mb(size_bytes: int) -> float:
    return size_bytes / (1024 * 1024)


rows = []

for label, image_name in images.items():
    size_bytes = get_image_size_bytes(image_name)

    rows.append(
        {
            "image_type": label,
            "image_name": image_name,
            "size_bytes": size_bytes,
            "size_mb": round(bytes_to_mb(size_bytes), 2) if size_bytes is not None else None,
        }
    )

size_report_df = pd.DataFrame(rows)

print("Docker image size comparison:")
print(size_report_df.to_string(index=False))

naive_size = size_report_df.loc[
    size_report_df["image_type"] == "naive", "size_bytes"
].iloc[0]

optimized_size = size_report_df.loc[
    size_report_df["image_type"] == "optimized", "size_bytes"
].iloc[0]

if naive_size is not None and optimized_size is not None:
    reduction_bytes = naive_size - optimized_size
    reduction_mb = bytes_to_mb(reduction_bytes)
    reduction_percent = (reduction_bytes / naive_size) * 100 if naive_size > 0 else 0

    result_text = (
        f"The optimized image is smaller by **{reduction_mb:.2f} MB**, "
        f"which is a **{reduction_percent:.2f}%** reduction compared with the naive image."
    )
else:
    reduction_mb = None
    reduction_percent = None

    result_text = (
        "Image sizes could not be fully measured because one or both Docker images "
        "were not available locally. Build both images before measuring exact sizes."
    )

report_text = f"""
# Docker Image Size Report

| Image Type | Image Name | Size MB |
|---|---|---:|
| Naive | airbnb-serving:naive | {size_report_df.loc[size_report_df["image_type"] == "naive", "size_mb"].iloc[0]} |
| Optimized | airbnb-serving:optimized | {size_report_df.loc[size_report_df["image_type"] == "optimized", "size_mb"].iloc[0]} |

## Result

{result_text}

## Explanation

The naive Docker image uses the full Python base image and copies the entire project directory into the container.

The optimized Docker image improves this by:

- using `python:3.11-slim`
- copying dependency files before source code for better Docker layer caching
- excluding notebooks, datasets, local artifacts, virtual environments, and cache folders through `.dockerignore`
- copying only the application source code needed for serving

This makes the optimized image more suitable for deployment because it is smaller, faster to transfer, and contains fewer unnecessary files.
""".strip() + "\n"

# This exact filename is required by the final proof cell.
expected_report_path = REPORT_DIR / "docker_size_report.md"
expected_report_path.write_text(report_text)

# Optional compatibility copy from the earlier filename, not required by final proof.
legacy_report_path = REPORT_DIR / "docker_image_size_report.md"
legacy_report_path.write_text(report_text)

print("\nCreated:", expected_report_path)
print("Created:", legacy_report_path)

Docker image size comparison:
image_type               image_name  size_bytes  size_mb
     naive     airbnb-serving:naive   656248784   625.85
 optimized airbnb-serving:optimized   294281844   280.65

Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/reports/docker_size_report.md
Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/reports/docker_image_size_report.md


### 3.4 Docker Compose

In [23]:
# TODO 3.4
# Create docker-compose.yml, .env.example, and .gitignore.
#
# docker-compose.yml runs the optimized Docker image and mounts the local
# MLflow fallback model folder so the service can still work if online MLflow
# artifact download fails.

(PROJECT / "docker-compose.yml").write_text(textwrap.dedent("""
services:
  airbnb-serving:
    build:
      context: .
      dockerfile: Dockerfile
    image: airbnb-serving:optimized
    container_name: airbnb-serving-api
    env_file:
      - .env
    ports:
      - "8001:8000"
    volumes:
      - ./mlflow_artifacts:/app/mlflow_artifacts:ro
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://127.0.0.1:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 5
      start_period: 60s
""").strip() + "\n")


(PROJECT / ".env.example").write_text(textwrap.dedent("""
# MLflow configuration
MLFLOW_TRACKING_URI=http://185.50.38.163:33014
MLFLOW_TRACKING_USERNAME=your_mlflow_username
MLFLOW_TRACKING_PASSWORD=your_mlflow_password

# Selected model from HW02
MLFLOW_EXPERIMENT_NAME=qbc12_hw02_student_alireza_abouei
MLFLOW_RUN_ID=2f4885cc6b2b42baa62c279937ea4401
MODEL_URI=runs:/2f4885cc6b2b42baa62c279937ea4401/model

# Local fallback model path inside the Docker container.
# docker-compose.yml mounts ./mlflow_artifacts to /app/mlflow_artifacts
LOCAL_MODEL_PATH=mlflow_artifacts/v5_random_forest/model.joblib
""").strip() + "\n")


(PROJECT / ".gitignore").write_text(textwrap.dedent("""
# Python
__pycache__/
*.py[cod]
*.pyo
*.pyd
.pytest_cache/
.mypy_cache/
.ruff_cache/

# Virtual environments
.venv/
venv/
env/

# Jupyter
.ipynb_checkpoints/

# IDE
.idea/
.vscode/

# Environment secrets
.env

# Local data and artifacts
data/
mlruns/
mlartifacts/
artifacts/
outputs/
reports/
screenshots/

# Logs
*.log
logs/

# OS files
.DS_Store
Thumbs.db
""").strip() + "\n")

print("Created:", PROJECT / "docker-compose.yml")
print("Created:", PROJECT / ".env.example")
print("Created:", PROJECT / ".gitignore")

Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/docker-compose.yml
Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/.env.example
Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/.gitignore


In [24]:
# Docker Compose smoke test
!docker compose up -d

import time, requests
time.sleep(8)  # wait for model to load from MLflow

health = requests.get('http://localhost:8001/health')
assert health.status_code == 200, f'Failed: {health.text}'
print('Docker Compose health check passed:', health.json())

!docker compose down

[+] up 0/1
 ⠋ Network hw03_a_default Creating                                          0.1s
[+] up 1/2
 ✔ Network hw03_a_default       Created                                     0.1s
 ⠋ Container airbnb-serving-api Creating                                    0.1s
[+] up 1/2
 ✔ Network hw03_a_default       Created                                     0.1s
 ⠙ Container airbnb-serving-api Starting                                    0.2s
[+] up 1/2
 ✔ Network hw03_a_default       Created                                     0.1s
 ⠹ Container airbnb-serving-api Starting                                    0.3s
[+] up 1/2
 ✔ Network hw03_a_default       Created                                     0.1s
 ⠸ Container airbnb-serving-api Starting                                    0.4s
[+] up 1/2
 ✔ Network hw03_a_default       Created                                     0.1s
 ⠼ Container airbnb-serving-api Starting                                    0.5s
[+] up 1/2
 ✔ Network hw03_a_default       

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

---
## Part 4 — Kubernetes Manifests

Kubernetes is the standard way to run containers in production at scale.

You do not need a real cluster for this homework. The deliverable is correct YAML files that a cluster could apply.

Key concepts you will use:

| Concept | What it does |
|---|---|
| **Pod** | Runs your container |
| **Deployment** | Manages multiple identical Pods, handles restarts |
| **Service** | Stable network endpoint that routes traffic to Pods |
| **Secret** | Stores sensitive values like passwords, not plaintext in YAML |
| **readinessProbe** | Tells Kubernetes when a Pod is ready to receive traffic |
| **resource limits** | Prevents one Pod from consuming all server memory |

### 4.1 Deployment

In [25]:
# TODO 4.1
# Create Kubernetes Deployment manifest.
#
# This Deployment runs the optimized FastAPI Docker image.
# It defines:
# - 2 replicas
# - container port 8000
# - MLflow/model environment variables
# - liveness and readiness probes
# - basic CPU/memory requests and limits

K8S_DIR = PROJECT / "k8s"
K8S_DIR.mkdir(parents=True, exist_ok=True)

(K8S_DIR / "deployment.yaml").write_text(textwrap.dedent("""
apiVersion: apps/v1
kind: Deployment
metadata:
  name: airbnb-serving-deployment
  labels:
    app: airbnb-serving
spec:
  replicas: 2
  selector:
    matchLabels:
      app: airbnb-serving
  template:
    metadata:
      labels:
        app: airbnb-serving
    spec:
      containers:
        - name: airbnb-serving-api
          image: airbnb-serving:optimized
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000

          env:
            - name: MLFLOW_TRACKING_URI
              value: "http://185.50.38.163:33014"

            - name: MLFLOW_EXPERIMENT_NAME
              value: "qbc12_hw02_student_alireza_abouei"

            - name: MLFLOW_RUN_ID
              value: "2f4885cc6b2b42baa62c279937ea4401"

            - name: MODEL_URI
              value: "runs:/2f4885cc6b2b42baa62c279937ea4401/model"

            - name: MLFLOW_TRACKING_USERNAME
              valueFrom:
                secretKeyRef:
                  name: mlflow-credentials
                  key: username
                  optional: true

            - name: MLFLOW_TRACKING_PASSWORD
              valueFrom:
                secretKeyRef:
                  name: mlflow-credentials
                  key: password
                  optional: true

          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 30
            periodSeconds: 15
            timeoutSeconds: 5
            failureThreshold: 5

          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 60
            periodSeconds: 30
            timeoutSeconds: 5
            failureThreshold: 5

          resources:
            requests:
              cpu: "250m"
              memory: "512Mi"
            limits:
              cpu: "1"
              memory: "1Gi"
""").strip() + "\n")

print("Created:", K8S_DIR / "deployment.yaml")

Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/k8s/deployment.yaml


### 4.2 Service

In [26]:
# TODO 4.2
# Create Kubernetes Service manifest.
#
# This Service exposes the FastAPI pods created by the Deployment.
# It maps:
# - service port 80
# - container target port 8000
#
# Type is ClusterIP because this is the safest/default internal Kubernetes service type.
# For local testing, use kubectl port-forward.

K8S_DIR = PROJECT / "k8s"
K8S_DIR.mkdir(parents=True, exist_ok=True)

(K8S_DIR / "service.yaml").write_text(textwrap.dedent("""
apiVersion: v1
kind: Service
metadata:
  name: airbnb-serving-service
  labels:
    app: airbnb-serving
spec:
  type: ClusterIP
  selector:
    app: airbnb-serving
  ports:
    - name: http
      protocol: TCP
      port: 80
      targetPort: 8000
""").strip() + "\n")

print("Created:", K8S_DIR / "service.yaml")

Created: /home/alireza/PycharmProjects/Quera-Homeworks/HW03_A/k8s/service.yaml


### 4.3 Conceptual questions

Answer these in the markdown cell below. One or two sentences each is enough.

**TODO 4.3 — Answer here:**

**Q1.** We set `replicas: 2` instead of 1. What happens to traffic if one Pod crashes while replicas is 1 vs 2?

A: If replicas is 1 and the only Pod crashes, the service has no healthy backend and traffic will fail until Kubernetes restarts the Pod. If replicas is 2, Kubernetes can continue routing traffic to the remaining healthy Pod while the crashed Pod is restarted.

**Q2.** The `readinessProbe` has `initialDelaySeconds: 15`. Why do we need a delay specifically for this service?

A: This service needs time to start FastAPI and load the ML model from MLflow or from the local fallback artifact. Without an initial delay, Kubernetes may check readiness too early and mark the Pod as not ready before the model loading process has finished.

**Q3.** Why do we store credentials in a Kubernetes Secret instead of writing them directly in `deployment.yaml`?

A: Credentials should not be written directly in `deployment.yaml` because that file is usually committed to Git and shared with others. Kubernetes Secrets separate sensitive values from the application manifest and make it safer to manage usernames, passwords, and tokens.

**Q4.** What is the difference between `ClusterIP` and `LoadBalancer` service types? When would you use each?

A: `ClusterIP` exposes the service only inside the Kubernetes cluster, so it is useful for internal services or local testing with port forwarding. `LoadBalancer` exposes the service externally through a cloud load balancer, so it is used when outside users or systems need direct access to the API.


---
## Final Proof

If this cell fails, HW03 is not complete.

In [27]:
required_files = [
    'src/airbnb_serving/__init__.py',
    'src/airbnb_serving/schema.py',
    'src/airbnb_serving/predictor.py',
    'src/airbnb_serving/app.py',
    'pyproject.toml',
    'requirements.txt',
    'Dockerfile',
    'Dockerfile.naive',
    'docker-compose.yml',
    '.env.example',
    '.dockerignore',
    'k8s/deployment.yaml',
    'k8s/service.yaml',
    'reports/docker_size_report.md',
]

missing = [f for f in required_files if not (PROJECT / f).exists()]
assert not missing, f'Missing files:\n' + '\n'.join(missing)

# Check .env is gitignored
gitignore = (PROJECT / '.gitignore').read_text() if (PROJECT / '.gitignore').exists() else ''
assert '.env' in gitignore, '.env must be in .gitignore'

# Check Dockerfile does not copy .env
for df_name in ['Dockerfile', 'Dockerfile.naive']:
    content = (PROJECT / df_name).read_text()
    assert 'COPY .env' not in content, f'Do not copy .env in {df_name}'

# Check schema.py has actual content
schema_content = (PROJECT / 'src/airbnb_serving/schema.py').read_text()
assert 'BaseModel' in schema_content, 'schema.py must define Pydantic models'

# Check app.py has endpoints
app_content = (PROJECT / 'src/airbnb_serving/app.py').read_text()
assert '/health' in app_content, 'app.py must have /health endpoint'
assert '/predict' in app_content, 'app.py must have /predict endpoint'
assert 'batch' in app_content, 'app.py must have /predict/batch endpoint'

print('All required files present.')
print('No credential leaks detected.')
print('HW03 proof passed.')

All required files present.
No credential leaks detected.
HW03 proof passed.


## Screenshots required

Add these to the `screenshots/` folder before submitting:

- `screenshots/health_endpoint.png` — GET /health response
- `screenshots/predict_endpoint.png` — POST /predict response
- `screenshots/batch_endpoint.png` — POST /predict/batch response
- `screenshots/fastapi_docs.png` — auto-generated docs at /docs
- `screenshots/docker_image_sizes.png` — output of `docker images` showing both image sizes

## Commit

```bash
git add .
git commit -m "HW03 model serving and deployment"
git push
```